In [ ]:
import pandas as pd

# =========================
# Config
# =========================
MB_PATH = "canonical_musicbrainz_data.csv"
HISTORY_PATH = "music_history.xlsx"

MB_KEY_COL = "artist_mbids"
MB_NAME_COL = "artist_credit_name"

HISTORY_ARTIST_COL_CANDIDATES = ["artist MBID"]

OUT_OVERLAP_LOOKUP_CSV = (
    "overlapped_artists_with_names.csv"  # artist_mbids + artist_credit_name
)
OUT_MB_OVERLAP_CSV = (
    "musicbrainz_artist_overlap.csv"  # MB rows restricted to overlapped artists
)
OUT_HISTORY_OVERLAP_XLSX = (
    "history_artist_overlap.xlsx"  # History rows restricted to overlapped artists
)

# =========================
# Load
# =========================
df_mb = pd.read_csv(MB_PATH)
df_history = pd.read_excel(HISTORY_PATH)

# =========================
# Normalize history key col
# =========================
# Detects the correct artist MBID column in the history dataset
hist_artist_col = None
for c in HISTORY_ARTIST_COL_CANDIDATES:
    if c in df_history.columns:
        hist_artist_col = c
        break

if hist_artist_col is None:
    raise KeyError(
        f"Couldn't find artist MBID column in music_history.xlsx. "
        f"Looked for: {HISTORY_ARTIST_COL_CANDIDATES}. "
        f"Available columns: {list(df_history.columns)}"
    )

# Standardizes column names for consistent merging
if hist_artist_col != MB_KEY_COL:
    df_history = df_history.rename(columns={hist_artist_col: MB_KEY_COL})

# Validate required MB columns exist
missing = [c for c in [MB_KEY_COL, MB_NAME_COL] if c not in df_mb.columns]

# Raises clear errors to prevent downstream data issues
if missing:
    raise KeyError(
        f"Missing required columns in musicbrainz_sample_5pct.csv: {missing}. "
        f"Available columns: {list(df_mb.columns)}"
    )

# =========================
# Clean keys (strip, string) + drop null/blank
# =========================
df_mb[MB_KEY_COL] = df_mb[MB_KEY_COL].astype("string").str.strip()
df_history[MB_KEY_COL] = df_history[MB_KEY_COL].astype("string").str.strip()

df_mb = df_mb[df_mb[MB_KEY_COL].notna() & (df_mb[MB_KEY_COL] != "")]
df_history = df_history[df_history[MB_KEY_COL].notna() & (df_history[MB_KEY_COL] != "")]

# =========================
# Build unique artist -> name mapping from MB
# =========================
# Counts unique names per artist ID
# drop duplicates
mb_artist_lookup = (
    df_mb[[MB_KEY_COL, MB_NAME_COL]]
    .dropna(subset=[MB_NAME_COL])
    .drop_duplicates(subset=[MB_KEY_COL])
)

# Optional sanity check: detect violations (same artist_mbids mapped to multiple names)
# Identifies whether artist IDs mapped to multiple names
# Prints warnings for data integrity violations
name_counts = df_mb.groupby(MB_KEY_COL)[MB_NAME_COL].nunique(dropna=True)
violations = name_counts[name_counts > 1]
if len(violations) > 0:
    print(
        f"WARNING: {len(violations):,} artist_mbids map to multiple artist_credit_name values."
    )
    print("Example problematic artist_mbids:", violations.head(10).index.tolist())


# =========================
# Compute overlap of unique artist_mbids
# =========================
mb_unique_ids = df_mb[[MB_KEY_COL]].drop_duplicates()
hist_unique_ids = df_history[[MB_KEY_COL]].drop_duplicates()

overlap_ids = mb_unique_ids.merge(hist_unique_ids, on=MB_KEY_COL, how="inner")

# Attach artist_credit_name (from MB lookup)
overlap_lookup = overlap_ids.merge(
    mb_artist_lookup,
    on=MB_KEY_COL,
    how="left",
    validate="1:1",  # overlap_ids should have unique artist_mbids
)

# =========================
# Summary
# =========================
print("=== Artist MBID Overlap Summary ===")
print(f"MB rows (non-null {MB_KEY_COL}): {len(df_mb):,}")
print(f"History rows (non-null {MB_KEY_COL}): {len(df_history):,}")
print(f"Unique artists in MB: {df_mb[MB_KEY_COL].nunique():,}")
print(f"Unique artists in history: {df_history[MB_KEY_COL].nunique():,}")
print(f"Overlapping unique artists: {len(overlap_lookup):,}")
print(
    f"Missing names in overlap (should be small): {overlap_lookup[MB_NAME_COL].isna().sum():,}"
)

# =========================
# Save overlapped artist ids + names
# =========================
overlap_lookup.to_csv(OUT_OVERLAP_LOOKUP_CSV, index=False)
print(f"Saved overlapped artist IDs + names -> {OUT_OVERLAP_LOOKUP_CSV}")

# =========================
# Filter original datasets to overlapped artists (for EDA)
# =========================
overlap_set = set(overlap_lookup[MB_KEY_COL])

df_mb_overlap = df_mb[df_mb[MB_KEY_COL].isin(overlap_set)]
df_history_overlap = df_history[df_history[MB_KEY_COL].isin(overlap_set)]

df_mb_overlap.to_csv(OUT_MB_OVERLAP_CSV, index=False)
df_history_overlap.to_excel(OUT_HISTORY_OVERLAP_XLSX, index=False)

print(f"Saved MB overlap rows -> {OUT_MB_OVERLAP_CSV}")
print(f"Saved history overlap rows -> {OUT_HISTORY_OVERLAP_XLSX}")